In [2]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
load_dotenv()
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
)


In [5]:
#start with state formation
class BatsmanState(TypedDict):
        runs:int
        balls:int
        fours:int
        sixes:int

        sr:float
        balls_per_boundary:float
        boundary_percent:float
        summary:str
        


In [10]:
def calculate_sr(state:BatsmanState)->BatsmanState:
    runs = state["runs"]
    balls = state["balls"]

    strike_rate = (runs/balls)*100
    return {"sr":strike_rate}
    

def balls_per_boundary (state:BatsmanState)->BatsmanState:
    fours = state["fours"]
    sixes = state["sixes"]
    total_balls = state["balls"]

    balls_per_boundary = total_balls / (fours+sixes)
    return {"balls_per_boundary":balls_per_boundary}
    

def boundary_percent(state:BatsmanState)->BatsmanState:
    fours = state["fours"]*4
    sixes = state["sixes"]*6
    runs = state["runs"]

    boundary_percent = (fours+sixes)/runs*100
    return {"boundary_percent":boundary_percent}

def summary(state:BatsmanState)->BatsmanState:
    summary= f"""
    strike_rate = {state["sr"]}\n
    Balls_percent = {state["balls_per_boundary"]}\n
    Boundary_percent = {state["boundary_percent"]}\n
    """
    state["summary"]=summary
    return state






In [11]:
#state graph 
graph=StateGraph(BatsmanState)

#add nodes to the graph
graph.add_node("calculate_sr",calculate_sr)
graph.add_node("calculate_balls_per_boundary",balls_per_boundary)
graph.add_node("calculate_boundary_percent",boundary_percent)
graph.add_node("summary",summary)


#add edges to the graph
graph.add_edge(START,"calculate_sr")
graph.add_edge(START,"calculate_balls_per_boundary")
graph.add_edge(START,"calculate_boundary_percent")
graph.add_edge("calculate_sr","summary")
graph.add_edge("calculate_balls_per_boundary","summary")
graph.add_edge("calculate_boundary_percent","summary")
graph.add_edge("summary",END)

#compile
workflow = graph.compile()

In [12]:
print(workflow.invoke({"runs":90,"balls":50,"fours":8,"sixes":4}))

{'runs': 90, 'balls': 50, 'fours': 8, 'sixes': 4, 'sr': 180.0, 'balls_per_boundary': 4.166666666666667, 'boundary_percent': 62.22222222222222, 'summary': '\n    strike_rate = 180.0\n\n    Balls_percent = 4.166666666666667\n\n    Boundary_percent = 62.22222222222222\n\n    '}
